# Transcript (Text) Preprocessing

Cleans the Sinhala transcript text in `df_trim` (the audio-trimmed dataset from
`audio.ipynb`), adapting the design decisions from the team's existing 15-stage
Sinhala corpus-cleaning pipeline (`preprocess.md`) to ASR transcripts instead of
scraped web documents.

**Two things are deliberately different here vs. the source pipeline:**

1. **No document-level steps apply.** The corpus pipeline's boilerplate removal,
   document-frequency deduplication, and literal `\n`-marker repair all target
   problems specific to scraped multi-paragraph web documents. Transcripts here
   are already short, single-utterance strings pulled from structured
   audio-transcript pairs — none of that class of noise exists in this data.
2. **Code-mixed English is kept, not stripped.** The source pipeline's step 02
   deletes standalone English words because it's building a *Sinhala-only*
   tokenizer corpus. This project's audio pipeline already made the opposite
   call (`audio.ipynb`, "Remove pure-English rows" — code-mixed Sinhala/English
   rows are explicitly kept, since code-mixing is natural in Sri Lankan speech).
   So the final whitelist here allows Sinhala + Latin script, not Sinhala only.

What does carry over directly: NFC Unicode normalization first (order-independent,
establishes a consistent baseline), the Sinhala-specific character-handling
rules (combining marks, ZWJ conjunct preservation, unassigned-codepoint
rejection, the `isspace()` control-character trap), and a final whitelist +
alphabet-inspection pass as a categorical safety net rather than an
ever-growing list of one-off patches.

**Every transformation below is observed on a sample first** (raw text printed,
counts of how many rows would be affected) **before being applied**, and applied
into a new column rather than overwriting `text` in place, so the original is
never lost and each stage can be inspected independently.

In [23]:
import os
import re
import unicodedata
from collections import Counter

import pandas as pd

# Prefer the audio-trimmed output; fall back to the raw combined dataset if audio.ipynb
# hasn't been run to completion yet, so this notebook can still be developed independently.
_CANDIDATES = ["../../../../data/processed/combinedData_trimmed.parquet", "../../../../data/processed/combinedData1.parquet"]
_path = next((p for p in _CANDIDATES if os.path.exists(p)), None)
if _path is None:
    raise FileNotFoundError(f"None of {_CANDIDATES} found -- run audio.ipynb first or adjust the path.")

df_trim = pd.read_parquet(_path, columns=None)
print(f"Loaded {_path}  ->  {df_trim.shape}")
df_trim[["source_dataset", "text"]].head(5)

Loaded ../../../../data/processed/combinedData_trimmed.parquet  ->  (15407, 14)


,source_dataset,text
0,linga,මහවැලි ගඟට ගොස් ආපසු එන ගමනේදී
1,linga,උන්වහන්සේ කපාපු
2,linga,එය එතනින් අවසන් නොවී
3,linga,සිතින් අයහපතෙහි හැසිරීම නිසයි.
4,linga,එවන් ශ්‍රේෂ්ඨ ජාතියක් බිහි කිරීමට


## Observe raw transcripts

Before touching anything: sample raw text across sources, check length distribution,
and run a full alphabet inspection (same idea as the source pipeline's step 14, but
run *first* here as a baseline instead of only at the end) to see what's actually in
the data before deciding what needs cleaning.

In [24]:
N_SAMPLE = 5

print("=== Random sample of raw transcripts, by source ===")
for source, g in df_trim.groupby("source_dataset"):
    print(f"\n--- {source} ({len(g)} rows) ---")
    for t in g["text"].fillna("").sample(min(N_SAMPLE, len(g)), random_state=42):
        print(f"  {t!r}")

print("\n=== Transcript length (characters) ===")
print(df_trim["text"].fillna("").str.len().describe())

=== Random sample of raw transcripts, by source ===

--- bizbrains (977 rows) ---
  'right ඒ කියන්නේ ඔයා entrepreneur කෙනෙක් විදියට ඔයා කොහොමද market එකට ඔයාව පෙන්නන්නේ මොකද දැන් කාලේ'
  'Business සම්බන්ධයෙන්, සහ investment සම්බන්ධයෙන්, wealth plan සම්බන්ධයෙන් තමන් income එක manage කරන එක සම්බන්ධයෙන් දැවෙන ප්\u200dරශ්න හිතට වදින විදිහට ඔලුවට දාගන්න ඕන නම්, මෙන්න මේ number එකට අපිට කියන්න. අපි නිලේන්ද්\u200dර හරහා ඒ වගේම අපේ ටීම් එකේ ඉන්න'
  'ඒකෙදි Hard decision ගන්නව. අනිත් එක ඒ ඔයා එවන සල්ලි කොහෙටද වියදම් උනේ කියලා අහන්න ඔයාට අයිතියක්'
  'අරන් තියන්න කියල ඕක වෙන් නෑනෙ. හැමවෙලාවෙම තමන්ගෙ ආදායම මොකක් වුනත් follow කරන්න'
  'failure එක උනාට පස්සේ කතා කරල වැඩක් නෑ ඇත්තටම එතකොට කාටවත් අදාළත් නෑ ඒකියන්නෙ මොකද ඊට පස්සේ වෙන්නේ බැංකුවට තමන්ගේ තියෙන වත්කම් ටික සින්න වෙලා අර'

--- linga (11357 rows) ---
  'එකිනෙකට යටින්'
  'යනුවෙන් නම් කැරෙනවා'
  'හැබැයි පෙති නම් ඉල්ලන්නෙ නෑ'
  'ඊයේ දා කැබිතිගොල්ලෑව මහෙස්ත්\u200dරාත් අධිකරණයට ඉදිරිපත් කිරීමට නියමිතව තිබිණි'
  'එය ජනතා අවශ්\u200dයතා ඉටුකරන මාධ්\u2

In [25]:
SINHALA_START, SINHALA_END = 0x0D80, 0x0DFF
ZW_CHARS = {0x200C, 0x200D}  # ZWNJ, ZWJ -- required for Sinhala conjuncts, not disposable


def inspect_alphabet(texts, top_n=40):
    """Reusable version of the source pipeline's step 14: count every unique character
    across the given texts, with its Unicode codepoint/category/name, plus rollups by
    category so contamination (Cn unassigned, Cc control, foreign scripts) is visible
    at a glance without reading the full per-character table.
    """
    counter = Counter()
    for t in texts:
        counter.update(t)

    rows = []
    for ch, n in counter.items():
        cp = ord(ch)
        rows.append({
            "char": ch,
            "codepoint": f"U+{cp:04X}",
            "category": unicodedata.category(ch),
            "name": unicodedata.name(ch, "<unassigned>"),
            "count": n,
            "is_sinhala_block": SINHALA_START <= cp <= SINHALA_END,
        })
    report = pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)

    print(f"Unique characters: {len(report)}")
    print("\nBy category:")
    print(report.groupby("category")["count"].agg(["size", "sum"]).rename(columns={"size": "unique_chars", "sum": "total_occurrences"}))

    n_unassigned = (report["category"] == "Cn").sum()
    n_control = ((report["category"] == "Cc") & (report["char"] != "\n")).sum()
    n_space_like = (report["category"] == "Zs").sum()
    print(f"\nUnassigned (Cn) codepoints: {n_unassigned}  <- encoding corruption if > 0")
    print(f"Stray control (Cc, excl. \\n) codepoints: {n_control}  <- corruption if > 0")
    print(f"Distinct space-like (Zs) characters: {n_space_like}  <- should be 1 after whitespace cleanup")

    return report


print("=== Baseline alphabet report (raw text, before any cleaning) ===")
baseline_report = inspect_alphabet(df_trim["text"].fillna(""))
baseline_report.head(40)

=== Baseline alphabet report (raw text, before any cleaning) ===
Unique characters: 158

By category:
          unique_chars  total_occurrences
category                                 
Cc                   1                  1
Cf                   2               3754
Ll                  26              54899
Lo                  55             347603
Lu                  26               1247
Mc                  14              75395
Mn                   5             105157
Nd                  10               1547
Pd                   2                 47
Pf                   2                  8
Pi                   2                  4
Po                  10               7520
Sc                   1                  1
Zs                   2             102946

Unassigned (Cn) codepoints: 0  <- encoding corruption if > 0
Stray control (Cc, excl. \n) codepoints: 0  <- corruption if > 0
Distinct space-like (Zs) characters: 2  <- should be 1 after whitespace cleanup


,char,codepoint,category,name,count,is_sinhala_block
0,,U+0020,Zs,SPACE,102940,False
1,්,U+0DCA,Mn,SINHALA SIGN AL-LAKUNA,47627,True
2,න,U+0DB1,Lo,SINHALA LETTER DANTAJA NAYANNA,47065,True
3,ි,U+0DD2,Mn,SINHALA VOWEL SIGN KETTI IS-PILLA,35576,True
4,ක,U+0D9A,Lo,SINHALA LETTER ALPAPRAANA KAYANNA,33740,True
5,ව,U+0DC0,Lo,SINHALA LETTER VAYANNA,29376,True
6,ය,U+0DBA,Lo,SINHALA LETTER YAYANNA,25297,True
7,ම,U+0DB8,Lo,SINHALA LETTER MAYANNA,24812,True
8,ා,U+0DCF,Mc,SINHALA VOWEL SIGN AELA-PILLA,24533,True
9,ත,U+0DAD,Lo,SINHALA LETTER ALPAPRAANA TAYANNA,22827,True


## Step 1 — Unicode normalization (NFC)

Same rationale as the source pipeline's step 00: different scrapers/input methods can
represent the same visible Sinhala character sequence with different underlying
codepoint sequences (e.g. a precomposed vs. decomposed combining-mark sequence). NFC
normalization makes every occurrence canonical, so every later regex/character-class
check in this notebook can assume a single consistent representation. Runs first,
before any other step, for the same reason it does in the source pipeline: it doesn't
touch plain ASCII and has no ordering dependency on anything else.

Observe how many rows actually change under NFC before applying it.

In [26]:
raw_text = df_trim["text"].fillna("")
nfc_preview = raw_text.apply(lambda t: unicodedata.normalize("NFC", t))

changed_mask = nfc_preview != raw_text
print(f"Rows that change under NFC normalization: {changed_mask.sum()} out of {len(raw_text)}")

for idx in df_trim[changed_mask].index[:5]:
    print(f"\n[{idx}] before: {raw_text.loc[idx]!r}")
    print(f"[{idx}] after:  {nfc_preview.loc[idx]!r}")

Rows that change under NFC normalization: 1 out of 15407

[14441] before: 'එතකොට අර Life style එක maintain කරගෙන යන්න පුළුවන්. අනිත් එක දරැවොන්ගෙ education එකට අපි කෙටි කාලින investment plan දාන්න පුලුවන්. ඔයා mask කීපයක් කරනවා'
[14441] after:  'එතකොට අර Life style එක maintain කරගෙන යන්න පුළුවන්. අනිත් එක දරැවොන්ගෙ education එකට අපි කෙටි කාලින investment plan දාන්න පුලුවන්. ඔයා mask කීපයක් කරනවා'


In [27]:
df_trim["text_v1_nfc"] = nfc_preview
print("Applied -- df_trim['text_v1_nfc'] added. Original 'text' column left untouched.")

Applied -- df_trim['text_v1_nfc'] added. Original 'text' column left untouched.


## Step 2 — Whitespace cleanup

Collapses repeated spaces/tabs, converts non-breaking spaces (`U+00A0`) and other
`Zs`-category space variants to a regular ASCII space, and trims leading/trailing
whitespace. Deliberately does **not** use `str.isspace()` to decide what counts as
whitespace — per the source pipeline's design notes, `isspace()` returns `True` for
some ASCII control characters (e.g. `U+001F`) that are corruption artifacts, not
real whitespace. Only actual Unicode space separators (`Zs` category) plus tab and
newline are treated as collapsible whitespace here.

Observe how many rows have non-standard whitespace before collapsing it.

In [28]:
def is_collapsible_space(ch):
    return unicodedata.category(ch) == "Zs" or ch in ("\t", "\n", "\r")


def has_nonstandard_whitespace(t):
    return any(is_collapsible_space(ch) and ch != " " for ch in t) or t != t.strip() or "  " in t


nonstd_ws_mask = df_trim["text_v1_nfc"].apply(has_nonstandard_whitespace)
print(f"Rows with non-standard whitespace (repeated spaces, non-breaking space, "
      f"leading/trailing whitespace): {nonstd_ws_mask.sum()} out of {len(df_trim)}")

for idx in df_trim[nonstd_ws_mask].index[:5]:
    print(f"[{idx}] {df_trim.loc[idx, 'text_v1_nfc']!r}")

Rows with non-standard whitespace (repeated spaces, non-breaking space, leading/trailing whitespace): 159 out of 15407
[11413] 'ඔන්න ආපහු අපි අමු අපේ site එකට. ගිහින් කලින් වගේම custom HTML එකක්  open කරලා'
[11468] 'මේ console ගොඩක් සිහින් නිසා පොඩි තල්ලුවකදි  වුනත් අනිත් පැත්තට වැටෙන්න පුළුවන්'
[11479] 'ගමකට ගහන්ඩ කතාව. පාන් බාගයක් හරියට කාගන්න පුළුවන්ද ?  බත් එකක් කාලා වතුර එකක් බොන්න බෑ ඉස්මුරුත්තාවට එනවා'
[11488] 'හැබැයි ඒගොල්ලන්ගේ අපේක්ෂාවන් මෙතන තියෙනවා. ඒ කියන්නේ 95 ඉඳලා තමයි මේ අරගල යන්නේ ගුරු විදුහල්පති වැටුප් විශමතාවය. එතකොට ඒ විෂමතාවය address කරනවා  නම්'
[11517] 'ඊට වඩා මුදලක්  generate කරන බිස්නස් එකක් build කරගන්න ඕනනම් කරුණාකරලා මුලින්ම device එක මාරු කරගන්න'


In [29]:
def clean_whitespace(t):
    t = "".join(" " if (is_collapsible_space(ch) and ch != "\n") else ch for ch in t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n+", "\n", t)
    return t.strip()


df_trim["text_v2_ws"] = df_trim["text_v1_nfc"].apply(clean_whitespace)

print("Applied -- df_trim['text_v2_ws'] added.")
for idx in df_trim[nonstd_ws_mask].index[:5]:
    print(f"\n[{idx}] before: {df_trim.loc[idx, 'text_v1_nfc']!r}")
    print(f"[{idx}] after:  {df_trim.loc[idx, 'text_v2_ws']!r}")

Applied -- df_trim['text_v2_ws'] added.

[11413] before: 'ඔන්න ආපහු අපි අමු අපේ site එකට. ගිහින් කලින් වගේම custom HTML එකක්  open කරලා'
[11413] after:  'ඔන්න ආපහු අපි අමු අපේ site එකට. ගිහින් කලින් වගේම custom HTML එකක් open කරලා'

[11468] before: 'මේ console ගොඩක් සිහින් නිසා පොඩි තල්ලුවකදි  වුනත් අනිත් පැත්තට වැටෙන්න පුළුවන්'
[11468] after:  'මේ console ගොඩක් සිහින් නිසා පොඩි තල්ලුවකදි වුනත් අනිත් පැත්තට වැටෙන්න පුළුවන්'

[11479] before: 'ගමකට ගහන්ඩ කතාව. පාන් බාගයක් හරියට කාගන්න පුළුවන්ද ?  බත් එකක් කාලා වතුර එකක් බොන්න බෑ ඉස්මුරුත්තාවට එනවා'
[11479] after:  'ගමකට ගහන්ඩ කතාව. පාන් බාගයක් හරියට කාගන්න පුළුවන්ද ? බත් එකක් කාලා වතුර එකක් බොන්න බෑ ඉස්මුරුත්තාවට එනවා'

[11488] before: 'හැබැයි ඒගොල්ලන්ගේ අපේක්ෂාවන් මෙතන තියෙනවා. ඒ කියන්නේ 95 ඉඳලා තමයි මේ අරගල යන්නේ ගුරු විදුහල්පති වැටුප් විශමතාවය. එතකොට ඒ විෂමතාවය address කරනවා  නම්'
[11488] after:  'හැබැයි ඒගොල්ලන්ගේ අපේක්ෂාවන් මෙතන තියෙනවා. ඒ කියන්නේ 95 ඉඳලා තමයි මේ අරගල යන්නේ ගුරු විදුහල්පති වැටුප් විශමතාවය. එතකොට ඒ විෂමතාවය address ක

## Step 3 — Punctuation cleanup

Removes empty bracket leftovers (`()`, `[ ]`) and collapses runs of repeated or mixed
punctuation (`...`, `?!`, `--`) down to a single mark — same targeted cleanup as the
source pipeline's step 03/08. Explicitly excludes Sinhala combining marks (`Mn`/`Mc`
categories) and the zero-width joiner/non-joiner from being touched by the
punctuation regex, per the doc's warning that a naive "letters are protected,
everything else is disposable punctuation" approach silently mangles every word
with a vowel sign.

Observe how many rows are affected before applying.

In [30]:
EMPTY_BRACKETS_RE = re.compile(r"\(\s*\)|\[\s*\]|\{\s*\}")
REPEATED_PUNCT_RE = re.compile(r"([.,!?;:\-]){2,}")
# Typographic quotes -> ASCII equivalents, applied before anything else here.
# Deleting these (like other disallowed punctuation) would be safe for quote-mark
# usage (word boundaries already have spaces) but wrong for apostrophes sitting
# mid-word (e.g. a curly-quote possessive/contraction) -- space-replacing those
# splits the word in two. Normalizing to ASCII keeps the character instead.
SMART_QUOTES = {"‘": "'", "’": "'", "“": '"', "”": '"'}


def clean_punctuation(t):
    for smart, ascii_eq in SMART_QUOTES.items():
        t = t.replace(smart, ascii_eq)
    t = EMPTY_BRACKETS_RE.sub("", t)
    t = REPEATED_PUNCT_RE.sub(r"\1", t)
    return t


punct_preview = df_trim["text_v2_ws"].apply(clean_punctuation)
punct_changed_mask = punct_preview != df_trim["text_v2_ws"]
print(f"Rows changed by punctuation cleanup: {punct_changed_mask.sum()} out of {len(df_trim)}")

for idx in df_trim[punct_changed_mask].index[:5]:
    print(f"\n[{idx}] before: {df_trim.loc[idx, 'text_v2_ws']!r}")
    print(f"[{idx}] after:  {punct_preview.loc[idx]!r}")

Rows changed by punctuation cleanup: 42 out of 15407

[305] before: 'ඒ අය වෙනුවෙන්..'
[305] after:  'ඒ අය වෙනුවෙන්.'

[401] before: 'ඉබේ වෙච්චි දෙයක්..'
[401] after:  'ඉබේ වෙච්චි දෙයක්.'

[446] before: 'උපසිරැසි යෙදුවා..'
[446] after:  'උපසිරැසි යෙදුවා.'

[886] before: 'එළ එළ ජය වේවා!.'
[886] after:  'එළ එළ ජය වේවා.'

[894] before: 'ලිපිවල පෙන්නල දුන්න..'
[894] after:  'ලිපිවල පෙන්නල දුන්න.'


In [31]:
df_trim["text_v3_punct"] = punct_preview
print("Applied -- df_trim['text_v3_punct'] added.")

Applied -- df_trim['text_v3_punct'] added.


## Step 4 — Emoji detection and removal

Speech transcripts are far less likely to contain emoji than web text, but crowd-sourced
or scraped-caption sources can still pick some up. Removes emoji character sequences,
including multi-codepoint emoji joined with ZWJ (e.g. family/flag emoji) — same logic
as the source pipeline's step 06: a ZWJ is only stripped when it sits directly between
two emoji codepoints, so it's never confused with the ZWJ used inside Sinhala
conjuncts (which sits between two Sinhala consonants, not emoji).

Observe how many rows actually contain emoji before deciding whether this step is
even needed for this dataset.

In [32]:
# Common emoji Unicode blocks (pictographs, symbols, transport, flags, supplemental
# symbols, dingbats). Deliberately range-based rather than a package dependency, since
# only detection + stripping is needed here, not emoji-aware text shortening/aliasing.
EMOJI_RANGES = [
    (0x1F300, 0x1FAFF),  # misc symbols & pictographs, emoticons, transport, supplemental symbols
    (0x2600, 0x27BF),    # misc symbols, dingbats
    (0x1F1E6, 0x1F1FF),  # regional indicators (flag letters)
    (0x2190, 0x21FF),    # arrows (occasionally used decoratively, low risk here)
    (0xFE0F, 0xFE0F),    # variation selector-16 (emoji presentation)
]


def is_emoji(ch):
    cp = ord(ch)
    return any(lo <= cp <= hi for lo, hi in EMOJI_RANGES)


def has_emoji(t):
    return any(is_emoji(ch) for ch in t)


emoji_mask = df_trim["text_v3_punct"].apply(has_emoji)
print(f"Rows containing emoji: {emoji_mask.sum()} out of {len(df_trim)}")

for idx in df_trim[emoji_mask].index[:5]:
    print(f"[{idx}] {df_trim.loc[idx, 'text_v3_punct']!r}")

Rows containing emoji: 0 out of 15407


In [33]:
def remove_emoji(t):
    out = []
    for i, ch in enumerate(t):
        if is_emoji(ch):
            continue
        # Drop a ZWJ only when it sits directly between two emoji codepoints -- leaves
        # Sinhala-conjunct ZWJ (between two Sinhala consonants) completely untouched.
        if ord(ch) == 0x200D and i > 0 and i + 1 < len(t) and is_emoji(t[i - 1]) and is_emoji(t[i + 1]):
            continue
        out.append(ch)
    return clean_whitespace("".join(out))


df_trim["text_v4_emoji"] = df_trim["text_v3_punct"].apply(remove_emoji)

print("Applied -- df_trim['text_v4_emoji'] added.")
for idx in df_trim[emoji_mask].index[:5]:
    print(f"\n[{idx}] before: {df_trim.loc[idx, 'text_v3_punct']!r}")
    print(f"[{idx}] after:  {df_trim.loc[idx, 'text_v4_emoji']!r}")

Applied -- df_trim['text_v4_emoji'] added.


## Step 5 — Final character whitelist (safety net)

Same rationale as the source pipeline's step 13: the targeted steps above only
guarantee that the specific patterns they were built for are gone, not what
character set survives overall. This final pass is a categorical safety net against
anything unanticipated — leftover Tamil/Arabic/etc. script fragments,
private-use-area and unassigned codepoints (encoding corruption), stray symbols.

**Whitelist for this project (Sinhala + Latin, not Sinhala-only):**
- Sinhala Unicode block, assigned codepoints only (rejects `Cn` unassigned gaps like
  `U+0DB2`, per the doc's encoding-corruption warning)
- ZWJ / ZWNJ (`U+200D` / `U+200C`) — required for Sinhala conjuncts
- ASCII Latin letters (`A-Za-z`) and digits — kept, unlike the source pipeline,
  because code-mixed English is intentionally retained in this dataset
- A small fixed punctuation set: `. , ! ? ; : ' " ( ) - /`
- Normalized whitespace (single space, single newline)

Same visibility-based removal rule as the source doc: invisible/control characters
(`Cf`, `Cc`, `Cn`) are deleted outright (no replacement), since inserting a visible
space where an invisible character used to be would introduce a word break that
never existed. Visible disallowed characters (foreign scripts, stray symbols) are
replaced with a single space, so two words separated only by the removed character
don't fuse together.

Observe what this would strip (dry run) before applying, since it's the most
aggressive step in the pipeline.

In [34]:
ALLOWED_PUNCTUATION = set(".,!?;:'\"()-/")


def is_allowed_char(ch):
    cp = ord(ch)
    if SINHALA_START <= cp <= SINHALA_END:
        return unicodedata.category(ch) != "Cn"  # reject unassigned gaps in the block
    if cp in ZW_CHARS:
        return True
    if ch.isascii() and (ch.isalpha() or ch.isdigit()):
        return True
    if ch in ALLOWED_PUNCTUATION:
        return True
    if ch in (" ", "\n"):
        return True
    return False


def whitelist_dry_run(t):
    """Returns (would_change, disallowed_chars_found) without modifying anything."""
    disallowed = {ch for ch in t if not is_allowed_char(ch)}
    return bool(disallowed), disallowed


dry_run = df_trim["text_v4_emoji"].apply(whitelist_dry_run)
would_change_mask = dry_run.apply(lambda r: r[0])
all_disallowed = Counter()
for _, disallowed in dry_run:
    all_disallowed.update(disallowed)

print(f"Rows the whitelist would change: {would_change_mask.sum()} out of {len(df_trim)}")
print(f"\nDisallowed characters found (would be stripped), by frequency:")
for ch, n in all_disallowed.most_common(30):
    cp = ord(ch)
    print(f"  {ch!r}  U+{cp:04X}  {unicodedata.category(ch)}  {unicodedata.name(ch, '<unassigned>')}  x{n}")

for idx in df_trim[would_change_mask].index[:5]:
    print(f"\n[{idx}] {df_trim.loc[idx, 'text_v4_emoji']!r}")

Rows the whitelist would change: 24 out of 15407

Disallowed characters found (would be stripped), by frequency:
  '%'  U+0025  Po  PERCENT SIGN  x21
  'ª'  U+00AA  Lo  FEMININE ORDINAL INDICATOR  x1
  '–'  U+2013  Pd  EN DASH  x1
  '$'  U+0024  Sc  DOLLAR SIGN  x1

[2033] 'මෙහි මුද්\u200dරාව අභය හෝ වරද මුද්\u200dරාව යුතුªයැයි සැලකේ'

[5197] 'සැයු – මෙය මාගේ'

[11567] 'හෙන උත්තරේ සීයෙන් වැඩි කරාම ඔන්න උත්තරේ 86% දශම 6 7 ආසන්න අගයට වැටයුවම'

[11590] 'මේකේ ගිය මාසේ සහ ඊට කලින් මාසේ මට මතක විදියට අඩු ගානක් ආවේ ඊට කලින් එහෙම නැත්තන් මාසෙට $150'

[11890] 'deep seek වලට පාවිච්චි කරන්න අවශ්\u200dය වෙන්නේ එයාගේ parameters වලින් 5% ක විතර කොටසක් විතරයි'


In [35]:
INVISIBLE_CATEGORIES = {"Cf", "Cc", "Cn"}


def apply_whitelist(t):
    out = []
    for ch in t:
        if is_allowed_char(ch):
            out.append(ch)
        elif unicodedata.category(ch) in INVISIBLE_CATEGORIES:
            pass  # delete outright -- no replacement, avoids inserting a false word break
        else:
            out.append(" ")  # visible disallowed char -- replace with space, avoids word fusion
    return clean_whitespace("".join(out))


df_trim["text_v5_whitelist"] = df_trim["text_v4_emoji"].apply(apply_whitelist)

print("Applied -- df_trim['text_v5_whitelist'] added.")
for idx in df_trim[would_change_mask].index[:5]:
    print(f"\n[{idx}] before: {df_trim.loc[idx, 'text_v4_emoji']!r}")
    print(f"[{idx}] after:  {df_trim.loc[idx, 'text_v5_whitelist']!r}")

Applied -- df_trim['text_v5_whitelist'] added.

[2033] before: 'මෙහි මුද්\u200dරාව අභය හෝ වරද මුද්\u200dරාව යුතුªයැයි සැලකේ'
[2033] after:  'මෙහි මුද්\u200dරාව අභය හෝ වරද මුද්\u200dරාව යුතු යැයි සැලකේ'

[5197] before: 'සැයු – මෙය මාගේ'
[5197] after:  'සැයු මෙය මාගේ'

[11567] before: 'හෙන උත්තරේ සීයෙන් වැඩි කරාම ඔන්න උත්තරේ 86% දශම 6 7 ආසන්න අගයට වැටයුවම'
[11567] after:  'හෙන උත්තරේ සීයෙන් වැඩි කරාම ඔන්න උත්තරේ 86 දශම 6 7 ආසන්න අගයට වැටයුවම'

[11590] before: 'මේකේ ගිය මාසේ සහ ඊට කලින් මාසේ මට මතක විදියට අඩු ගානක් ආවේ ඊට කලින් එහෙම නැත්තන් මාසෙට $150'
[11590] after:  'මේකේ ගිය මාසේ සහ ඊට කලින් මාසේ මට මතක විදියට අඩු ගානක් ආවේ ඊට කලින් එහෙම නැත්තන් මාසෙට 150'

[11890] before: 'deep seek වලට පාවිච්චි කරන්න අවශ්\u200dය වෙන්නේ එයාගේ parameters වලින් 5% ක විතර කොටසක් විතරයි'
[11890] after:  'deep seek වලට පාවිච්චි කරන්න අවශ්\u200dය වෙන්නේ එයාගේ parameters වලින් 5 ක විතර කොටසක් විතරයි'


## Verify the output (final alphabet report)

Same check as the source pipeline's step 14, run again here on the cleaned text as
a final QA gate. Compare against the baseline report from the top of this notebook:
unique character count should have dropped sharply, `Cn`/stray `Cc` entries should
be gone, and exactly one space-like (`Zs`) character should remain.

In [36]:
print(f"=== Final alphabet report (cleaned text) ===")
final_report = inspect_alphabet(df_trim["text_v5_whitelist"])

print(f"\nUnique chars: {len(baseline_report)} (baseline) -> {len(final_report)} (cleaned)")

emptied_mask = (df_trim["text_v5_whitelist"].str.strip() == "") & (df_trim["text"].fillna("").str.strip() != "")
print(f"\nRows that became empty after cleaning (whole utterance was junk): {emptied_mask.sum()}")
for idx in df_trim[emptied_mask].index[:10]:
    print(f"  [{idx}] original: {df_trim.loc[idx, 'text']!r}")

final_report.head(40)

=== Final alphabet report (cleaned text) ===
Unique characters: 149

By category:
          unique_chars  total_occurrences
category                                 
Cc                   1                  1
Cf                   2               3754
Ll                  26              54899
Lo                  54             347602
Lu                  26               1247
Mc                  14              75394
Mn                   5             105157
Nd                  10               1547
Pd                   1                 46
Po                   9               7458
Zs                   1             102764

Unassigned (Cn) codepoints: 0  <- encoding corruption if > 0
Stray control (Cc, excl. \n) codepoints: 0  <- corruption if > 0
Distinct space-like (Zs) characters: 1  <- should be 1 after whitespace cleanup

Unique chars: 158 (baseline) -> 149 (cleaned)

Rows that became empty after cleaning (whole utterance was junk): 0


,char,codepoint,category,name,count,is_sinhala_block
0,,U+0020,Zs,SPACE,102764,False
1,්,U+0DCA,Mn,SINHALA SIGN AL-LAKUNA,47627,True
2,න,U+0DB1,Lo,SINHALA LETTER DANTAJA NAYANNA,47065,True
3,ි,U+0DD2,Mn,SINHALA VOWEL SIGN KETTI IS-PILLA,35576,True
4,ක,U+0D9A,Lo,SINHALA LETTER ALPAPRAANA KAYANNA,33740,True
5,ව,U+0DC0,Lo,SINHALA LETTER VAYANNA,29376,True
6,ය,U+0DBA,Lo,SINHALA LETTER YAYANNA,25297,True
7,ම,U+0DB8,Lo,SINHALA LETTER MAYANNA,24812,True
8,ා,U+0DCF,Mc,SINHALA VOWEL SIGN AELA-PILLA,24532,True
9,ත,U+0DAD,Lo,SINHALA LETTER ALPAPRAANA TAYANNA,22827,True


## Step 6 — Pure English transcripts (flag for removal)

This is a Sinhala ASR corpus; code-mixed Sinhala/English rows are intentionally kept
(see the whitelist above), but rows with **no Sinhala script at all** don't belong in
it -- same call already made for the OpenSLR-52 portion of this corpus in
`audio_openslr.ipynb` ("Remove pure-English clips"), applied here for consistency
across all three sources (Linga/YouTube/BizBrains). Flagged on the fully-cleaned
`text_v5_whitelist` column; actually dropped together with the emptied rows in the
commit step below.

In [37]:
ENGLISH_LETTER_RE = re.compile(r"[A-Za-z]")


def is_pure_english(t):
    has_sinhala = any(SINHALA_START <= ord(ch) <= SINHALA_END for ch in t)
    has_english = bool(ENGLISH_LETTER_RE.search(t))
    return (not has_sinhala) and has_english


pure_english_mask = df_trim["text_v5_whitelist"].apply(is_pure_english)
print(f"Pure English rows (no Sinhala script, has Latin letters): "
      f"{pure_english_mask.sum()} out of {len(df_trim)}")
df_trim.loc[pure_english_mask, ["source_dataset", "text_v5_whitelist"]].head(10)


Pure English rows (no Sinhala script, has Latin letters): 405 out of 15407


,source_dataset,text_v5_whitelist
5,linga,day offices
33,linga,magazines of the 90s
36,linga,android
58,linga,windows xp service pack 3 sp3
87,linga,full hindi movies with sinhala subtitle
90,linga,election department
97,linga,world trade center sri lanka
112,linga,nokla 525
153,linga,ethiopian
167,linga,harappa civiliation


## Commit cleaned text + drop emptied and pure-English rows

**Status: applied.** Promotes `text_v5_whitelist` -> `text`, drops rows that became
empty after cleaning and rows flagged as pure English above, recomputes `text_len`
to match the new text (was going stale otherwise -- it was computed on the original
raw text and never updated), and drops the intermediate `text_v1`...`text_v5_whitelist`
columns.

In [38]:
drop_mask = emptied_mask | pure_english_mask
df_trim = df_trim[~drop_mask].reset_index(drop=True)
df_trim["text"] = df_trim["text_v5_whitelist"]
df_trim["text_len"] = df_trim["text"].str.len()
df_trim = df_trim.drop(columns=["text_v1_nfc", "text_v2_ws", "text_v3_punct", "text_v4_emoji", "text_v5_whitelist"])
print(f"Dropped {emptied_mask.sum()} emptied rows and {pure_english_mask.sum()} pure-English rows "
      f"-- new shape: {df_trim.shape}")

out_path = "../../../../data/processed/combinedData_transcript_cleaned.parquet"
df_trim.to_parquet(out_path, index=False)
print(f"Saved cleaned dataset to {out_path}")


Dropped 0 emptied rows and 405 pure-English rows -- new shape: (15002, 14)
Saved cleaned dataset to ../../../../data/processed/combinedData_transcript_cleaned.parquet


## Verify the saved cleaned parquet

Reads `combinedData_transcript_cleaned.parquet` back from disk (not the in-memory
`df_trim`) as a final sanity check that what got written matches what was intended.

In [39]:
cleaned_path = "../../../../data/processed/combinedData_transcript_cleaned.parquet"
df_clean = pd.read_parquet(cleaned_path)

print(f"Path: {cleaned_path}")
print(f"Shape: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns")
print(f"\nColumns and dtypes:")
print(df_clean.dtypes)
print(f"\nMemory usage: {df_clean.memory_usage(deep=True).sum() / 1e6:.2f} MB")

print(f"\nRows per source_dataset:")
print(df_clean["source_dataset"].value_counts())

print(f"\ntext_len summary:")
print(df_clean["text_len"].describe())

print(f"\nNull counts per column:")
print(df_clean.isnull().sum())

df_clean.head(5)

Path: ../../../../data/processed/combinedData_transcript_cleaned.parquet
Shape: 15002 rows x 14 columns

Columns and dtypes:
audio                 object
text                     str
source_dataset           str
orig_split               str
duration             float64
text_len               int64
vad_silence_frac     float64
vad_lead_sil_s       float64
vad_trail_sil_s      float64
vad_speech_s         float64
vad_n_segments         int64
duration_pre_trim    float64
trimmed_lead_s       float64
trimmed_trail_s      float64
dtype: object

Memory usage: 2077.54 MB

Rows per source_dataset:
source_dataset
linga        10959
youtube       3066
bizbrains      977
Name: count, dtype: int64

text_len summary:
count    15002.000000
mean        46.198973
std         43.033202
min          3.000000
25%         21.000000
50%         29.000000
75%         54.000000
max        415.000000
Name: text_len, dtype: float64

Null counts per column:
audio                    0
text                     0


,audio,text,source_dataset,orig_split,duration,text_len,vad_silence_frac,vad_lead_sil_s,vad_trail_sil_s,vad_speech_s,vad_n_segments,duration_pre_trim,trimmed_lead_s,trimmed_trail_s
0,b'RIFF$\x90\x01\x00WAVEfmt \x10\x00\x00\x00\x0...,මහවැලි ගඟට ගොස් ආපසු එන ගමනේදී,linga,NaN,3.2,30,0.535714,2.2,0.8,2.6,1,5.6,1.9,0.5
1,b'RIFF\xa4j\x01\x00WAVEfmt \x10\x00\x00\x00\x0...,උන්වහන්සේ කපාපු,linga,NaN,2.9,15,0.517241,0.8,0.7,1.4,1,2.9,0.0,0.0
2,b'RIFF$^\x01\x00WAVEfmt \x10\x00\x00\x00\x01\x...,එය එතනින් අවසන් නොවී,linga,NaN,2.8,20,0.428571,1.2,0.0,1.6,1,2.8,0.0,0.0
3,b'RIFF\xa4\xb5\x01\x00WAVEfmt \x10\x00\x00\x00...,සිතින් අයහපතෙහි හැසිරීම නිසයි.,linga,NaN,3.5,30,0.472727,1.4,1.2,2.9,1,5.5,1.1,0.9
4,b'RIFF$w\x01\x00WAVEfmt \x10\x00\x00\x00\x01\x...,එවන් ශ්‍රේෂ්ඨ ජාතියක් බිහි කිරීමට,linga,NaN,3.0,33,0.500000,1.2,0.4,1.6,1,3.2,0.2,0.0


## Export final dataset

Keeps the same format as the source parquet (`audio` as raw embedded WAV bytes, not
extracted files) -- just narrows `df_clean` down to the three columns needed and
writes it out as a new parquet into `model-development/data/final_dataset/`:
`audio` (bytes), `source_dataset` (`linga` / `bizbrains` / `youtube`), and `text`.

In [40]:
FINAL_DIR = "../../data/final_dataset"
os.makedirs(FINAL_DIR, exist_ok=True)

final_dataset = df_clean[["audio", "source_dataset", "text"]]

final_path = os.path.join(FINAL_DIR, "final_dataset.parquet")
final_dataset.to_parquet(final_path, index=False)

print(f"Saved final dataset ({len(final_dataset)} rows) to {final_path}")
print(f"\nRows per source_dataset:")
print(final_dataset["source_dataset"].value_counts())
final_dataset.head(5)

Saved final dataset (15002 rows) to ../../data/final_dataset/final_dataset.parquet

Rows per source_dataset:
source_dataset
linga        10959
youtube       3066
bizbrains      977
Name: count, dtype: int64


,audio,source_dataset,text
0,b'RIFF$\x90\x01\x00WAVEfmt \x10\x00\x00\x00\x0...,linga,මහවැලි ගඟට ගොස් ආපසු එන ගමනේදී
1,b'RIFF\xa4j\x01\x00WAVEfmt \x10\x00\x00\x00\x0...,linga,උන්වහන්සේ කපාපු
2,b'RIFF$^\x01\x00WAVEfmt \x10\x00\x00\x00\x01\x...,linga,එය එතනින් අවසන් නොවී
3,b'RIFF\xa4\xb5\x01\x00WAVEfmt \x10\x00\x00\x00...,linga,සිතින් අයහපතෙහි හැසිරීම නිසයි.
4,b'RIFF$w\x01\x00WAVEfmt \x10\x00\x00\x00\x01\x...,linga,එවන් ශ්‍රේෂ්ඨ ජාතියක් බිහි කිරීමට
